In [ ]:
import pandas as pd
import numpy as np

df= pd.read_csv('C:/Dumps/LTE_total/EUtranCellFDD.txt', engine='python',delimiter='\t', usecols=['UF', 'eNB', 'CELL','earfcndl', 'dlChannelBandwidth', 'cellSubscriptionCapacity',\
                                                                                  'cellCapMaxCellSubCap', 'cellCapMinCellSubCap', 'administrativeState'])

df.head()

In [ ]:
import pandas as pd
import numpy as np

# Certificando que as colunas relevantes são do tipo correto
df = df.astype({'dlChannelBandwidth': 'float', 
                'earfcndl': 'int', 
                'cellSubscriptionCapacity': 'float', 
                'cellCapMaxCellSubCap': 'float', 
                'cellCapMinCellSubCap': 'float'})

# Criando a coluna 'Regional'
df['Regional']= df['UF'].replace(['AM', 'AP', 'MA', 'PA', 'RR', 'BA', 'SE'], ['N', 'N', 'N', 'N', 'N', 'BASE', 'BASE'])

# Criando a coluna 'Banda'
df['Banda']= df['CELL'].str[0:1].replace(['T','Q', 'V', 'Z', 'U', 'P', 'Y', 'C', 'L', 'O'],
                                           ['2600','2600', '1800', '700', '2100', '2600', '850', '1800', '2300', '2300'])

# Inicializando as colunas de teste com um valor padrão
df['teste_Cellsubscriptioncapacity'] = 1
df['teste_cellCapMaxCellSubCap'] = 1
df['teste_cellCapMinCellSubCap'] = 1

# Definindo as regras para criar as colunas de baseline e teste
rules = [
    (5000, 475, 8500, 12740, 4240, 0),
    (10000, 200, 17000, 25500, 8500, 0),
    (10000, 549, 17000, 25500, 8500, 0),
    (10000, 1250, 17000, 25500, 8500, 0),
    (10000, 1800, 17000, 25500, 8500, 0),
    (10000, 1900, 17000, 25500, 8500, 0),
    (10000, 2800, 20000, 30000, 10000, 0),
    (10000, 9310, 12500, 18740, 6240, 0),
    (10000, 9510, 12500, 18740, 6240, 0),
    (15000, 425, 25500, 38240, 12740, 0),
    (15000, 525, 25500, 38240, 12740, 0),
    (15000, 1275, 25500, 38240, 12740, 0),
    (15000, 3375, 30000, 45000, 15000, 0),
    (20000, 450, 34000, 51000, 17000, 0),
    (20000, 500, 34000, 51000, 17000, 0),
    (20000, 1300, 34000, 51000, 17000, 0),
    (20000, 1750, 34000, 51000, 17000, 0),
    (20000, 1850, 34000, 51000, 17000, 0),
    (20000, 3200, 40000, 60000, 20000, 0),
    (20000, 3350, 40000, 60000, 20000, 0),
    (20000, 9460, 25000, 37500, 12500, 0),
]

# Aplicando as regras para criar as novas colunas
for bw, earfcn, csc_base, ccmax_base, ccmin_base, test_val in rules:
    mask = (df['dlChannelBandwidth'] == bw) & (df['earfcndl'] == earfcn)
    
    df.loc[mask, 'Cellsubscriptioncapacity_Baseline'] = csc_base
    df.loc[mask, 'cellCapMaxCellSubCap_Baseline'] = ccmax_base
    df.loc[mask, 'cellCapMinCellSubCap_Baseline'] = ccmin_base
    
    df.loc[:, 'teste_Cellsubscriptioncapacity'] = np.where(mask, np.where(df['cellSubscriptionCapacity'] == csc_base, test_val, 1), df['teste_Cellsubscriptioncapacity'])
    df.loc[:, 'teste_cellCapMaxCellSubCap'] = np.where(mask, np.where(df['cellCapMaxCellSubCap'] == ccmax_base, test_val, 1), df['teste_cellCapMaxCellSubCap'])
    df.loc[:, 'teste_cellCapMinCellSubCap'] = np.where(mask, np.where(df['cellCapMinCellSubCap'] == ccmin_base, test_val, 1), df['teste_cellCapMinCellSubCap'])

# Aplicando a condição WHERE e ORDER BY
filtered_df = df[
    (df['administrativeState'] == 'UNLOCKED') &
    ((df['teste_cellCapMaxCellSubCap'] + df['teste_cellCapMinCellSubCap'] + df['teste_Cellsubscriptioncapacity']) > 0) &
    (df['Cellsubscriptioncapacity_Baseline'] != 0)
]

sorted_df = filtered_df.sort_values(by=['Regional', 'UF', 'eNB', 'CELL'])

# Criando uma lista de todas as colunas com "Regional" como a primeira coluna
cols = ['Regional'] + [col for col in df if col != 'Regional']

# Reordenando as colunas do DataFrame
sorted_df = sorted_df[cols]

# Visualizando o resultado final
print(sorted_df)

sorted_df.to_excel('C:/Vivo Reports/LTE/LTE Parametros/Cellsubscriptioncapacity.xlsx', index= None)

In [ ]:
# import pandas as pd

# df= pd.read_excel('C:/Vivo Reports/5G/5G Parametros/percentual_5G.xlsx')
# df

# Subtrai 'Percentual_N' das outras colunas 'Percentual' e calcula o valor absoluto
diferenca_N = abs(df['Percentual_MG'] - df['Percentual_N'])
diferenca_BASE = abs(df['Percentual_MG'] - df['Percentual_BASE'])
diferenca_SP = abs(df['Percentual_MG'] - df['Percentual_SP'])

# Define um limiar para considerar a diferença 'significativa'
limiar = 30  # altere isso conforme necessário

# Encontra os nomes dos parâmetros onde a diferença é maior do que o limiar
parametros_diferentes_N = df['Parameter Name'][diferenca_N > limiar]
parametros_diferentes_BASE = df['Parameter Name'][diferenca_BASE > limiar]
parametros_diferentes_SP = df['Parameter Name'][diferenca_SP > limiar]

# Concatena todos os resultados
parametros_diferentes = pd.concat([parametros_diferentes_N, parametros_diferentes_BASE, parametros_diferentes_SP])

# Remove duplicatas
parametros_diferentes = parametros_diferentes.drop_duplicates()

parametros_diferentes.to_excel('C:/Camilao/diferenças_5G_W44.xlsx', index=None)
print('Done')

In [ ]:
print(True * 1)  # Output: 0